## 2. Chart: Exploring the data and creating visualisation
### 2.1 Load the clean data

In [1]:
import pandas as pd

clean_data = pd.read_csv("data/clean_data.csv", parse_dates=["month"], dtype={"coicop_code": str})
clean_data.head()

,month,geo,coicop_code,category,value
0,2019-01-01,Austria,00,All items,1.7
1,2019-02-01,Austria,00,All items,1.4
2,2019-03-01,Austria,00,All items,1.7
3,2019-04-01,Austria,00,All items,1.7
4,2019-05-01,Austria,00,All items,1.7


Columns in clean_data:
- month: the month of the value (Jan 2019 – Dec 2025)
- geo: the place (Ireland, Euro area, or one of the other 26 EU countries)
- coicop_code: the official category code (00, 01, 04, ZZZ3)
- category: the category name (All items, Food, Housing, Energy)
- value: annual price change (%), compared with the same month a year earlier

In [2]:
clean_data.shape

(9408, 5)

### 2.2 Exploratory chart 1 - Ireland vs Euro area (All items)

In [3]:
# Source: Plotly line charts, https://plotly.com/python/line-charts/
import plotly.express as px

# Filter only "All items" for Ireland and the Euro area
ireland_vs_euro = clean_data[(clean_data["category"] == "All items") & 
                        (clean_data["geo"].isin(["Ireland", "Euro area"]))]

chart1 = px.line(ireland_vs_euro, x="month", y="value", color="geo",
                title="Exploratory: Ireland vs Euro area, all items")
chart1.show()

What: Ireland vs the Euro area for All items, 2019–2025. Exploratory, not the final chart.

Why: To see the overall story before designing the final chart.

Result: Both peaked in 2022. Ireland reached 9.6% (in June-July 2022), a little below the Euro area at 10.6% (in October 2022). So overall, prices in Ireland did not rise faster than in the Euro area.

### 2.3 Exploratory chart 2 - Ireland vs Euro area by category

In [4]:
# Source: pandas group by, https://pandas.pydata.org/docs/user_guide/groupby.html
# and Plotly horizontal bar charts, https://plotly.com/python/horizontal-bar-charts/

# Keep only the crisis years (2022 and 2023)
crisis_year = clean_data[clean_data["month"].dt.year.isin([2022, 2023])]

# Average rate for each category and place
average = crisis_year.groupby(["category", "geo"])["value"].mean().unstack()

# Ireland minus Euro area: above 0 means prices rose faster in Ireland
gap = (average["Ireland"] - average["Euro area"]).reset_index(name="gap")
gap = gap.sort_values("gap")

chart2 = px.bar(gap, x="gap", y="category", orientation="h",
                title="Exploratory: Ireland minus Euro area, 2022-2023")
chart2.show()

What: The average gap between Ireland and the Euro area for each category in 2022-2023. Exploratory, not the final chart.

Why: To see which category rose faster in Ireland.

Result: Housing (+8.1) and Energy (+6.0) rose faster in Ireland. Food (-2.8) rose slower, and All items (-0.3) was about the same. Because each category tells a different story, the final chart needs a button for each one.

### 2.4 Band data - middle half of EU countries

In [5]:
# Keep only the other 26 EU countries (not Ireland and Euro area)
countries = clean_data[~clean_data["geo"].isin(["Euro area", "Ireland"])]

# For each category and month: bottom and top of the middle half of countries
low = countries.groupby(["category", "month"])["value"].quantile(0.25)
high = countries.groupby(["category", "month"])["value"].quantile(0.75)

band = pd.DataFrame({"low": low, "high": high}).reset_index()
band.head()

,category,month,low,high
0,All items,2019-01-01,1.050,2.075
1,All items,2019-02-01,1.150,2.250
2,All items,2019-03-01,1.225,2.600
3,All items,2019-04-01,1.500,2.625
4,All items,2019-05-01,1.025,2.575


What: For each category and month, the range where most of the other 26 EU countries were (the middle half).

Why: One grey area is cleaner than 26 grey lines. A few extreme countries (up to +114%) stretched the chart and hid the Ireland gap.

Result: 336 rows (4 categories × 84 months), with columns low and high.

### 2.5 Final chart - one category first (Housing)

In [6]:
# Source: Plotly continuous error bands, https://plotly.com/python/continuous-error-bars/
# and Plotly line charts, https://plotly.com/python/line-charts/

import plotly.graph_objects as go

# Test with just one category (Housing) first
category = "Housing"

# Prepare data for this category
band_one = band[band["category"] == category]
ireland_line = clean_data[(clean_data["category"] == category) & (clean_data["geo"] == "Ireland")]
euro_line = clean_data[(clean_data["category"] == category) & (clean_data["geo"] == "Euro area")]

final_chart = go.Figure()

# Grey band: bottom edge (no line), then top edge filled down to it
final_chart.add_trace(go.Scatter(x=band_one["month"], y=band_one["low"],
                                 line=dict(width=0), showlegend=False))
final_chart.add_trace(go.Scatter(x=band_one["month"], y=band_one["high"],
                                 line=dict(width=0), fill="tonexty",
                                 fillcolor="lightgrey", name="Most other EU countries"))

# Eurozone average: dashed dark blue line
final_chart.add_trace(go.Scatter(x=euro_line["month"], y=euro_line["value"], name="Eurozone average",
                                 line=dict(color="#1F3B73", width=2, dash="dash")))

# Ireland: thick green line
final_chart.add_trace(go.Scatter(x=ireland_line["month"], y=ireland_line["value"], name="Ireland",
                                 line=dict(color="#00843D", width=4)))

final_chart.update_layout(title="Housing: Ireland vs the rest of Europe",
                          yaxis_title="Annual price change (%)",
                          template="simple_white")
final_chart.show()

What: First version of the final chart for one category (Housing).

Why: To test the band and the two lines before adding all 4 categories.

Result: Ireland's green line is above the grey band in 2022–2023, so housing costs rose faster in Ireland than in most EU countries.

### 2.6 Final chart - all 4 categories with buttons

In [ ]:
# Source: Plotly custom buttons, https://plotly.com/python/custom-buttons/
# and Plotly continuous error bands, https://plotly.com/python/continuous-error-bars/

categories = ["All items", "Food", "Housing", "Energy"]

# Headline title for each button (numbers from Step 2.3)
titles = {
    "All items": "During the crisis, Irish prices rose about the same as the eurozone",
    "Food": "During the crisis, Irish food prices rose less than in any other EU country",
    "Housing": "During the crisis, Irish housing costs rose almost twice as fast as the eurozone",
    "Energy": "During the crisis, Irish energy prices rose faster than the eurozone",
}

final_chart = go.Figure()

# Same 4 lines as Step 2.5, once for each category
for category in categories:
    band_one = band[band["category"] == category]
    ireland_line = clean_data[(clean_data["category"] == category) & (clean_data["geo"] == "Ireland")]
    euro_line = clean_data[(clean_data["category"] == category) & (clean_data["geo"] == "Euro area")]

    # Only "All items" is shown when the chart opens
    show = (category == "All items")

    # Grey band, Eurozone average and Ireland (same as Step 2.5)
    final_chart.add_trace(go.Scatter(x=band_one["month"], y=band_one["low"],
                                     line=dict(width=0), showlegend=False, visible=show))
    final_chart.add_trace(go.Scatter(x=band_one["month"], y=band_one["high"],
                                     line=dict(width=0), fill="tonexty", fillcolor="lightgrey",
                                     name="Most other EU countries", visible=show))
    final_chart.add_trace(go.Scatter(x=euro_line["month"], y=euro_line["value"], name="Eurozone average",
                                     line=dict(color="#1F3B73", width=2, dash="dash"), visible=show))
    final_chart.add_trace(go.Scatter(x=ireland_line["month"], y=ireland_line["value"], name="Ireland",
                                     line=dict(color="#00843D", width=4), visible=show))

# One button per category
buttons = []
for i, category in enumerate(categories):
    # 16 lines in total (4 per category). Switch on only this category's 4 lines
    visible = [False] * 16
    visible[i * 4 : i * 4 + 4] = [True] * 4

    buttons.append(dict(label=category, method="update",
                        args=[{"visible": visible},
                              {"title.text": titles[category], "yaxis.autorange": True}]))

final_chart.update_layout(title=titles["All items"],
                          title_font_size=20,
                          yaxis_title="Annual price change (%)",
                          template="simple_white",
                          updatemenus=[dict(type="buttons", direction="right", buttons=buttons,
                                            x=0, xanchor="left", y=1.15)])

final_chart.show()

What: The final chart with all 4 categories and a button for each one.

Why: Lets the user switch between categories to see which ones drove the crisis. The title changes to state what each view shows.

Result: All items opens first. Clicking Food, Housing or Energy redraws the same chart with that category's data and title.

### 2.7 Styling of the final chart

In [8]:
# Source: Plotly shapes, https://plotly.com/python/horizontal-vertical-shapes/

# Shade the crisis years
final_chart.add_vrect(x0="2022-01-01", x1="2023-12-31", fillcolor="gold", opacity=0.12,
                      line_width=0, layer="below", annotation_text="Cost-of-living crisis")

# Line at 0%
final_chart.add_hline(y=0, line_color="grey", line_width=1)

# No hover for the grey band
final_chart.update_traces(hoverinfo="skip", selector=dict(showlegend=False))
final_chart.update_traces(hoverinfo="skip", selector=dict(name="Most other EU countries"))

# Source note under the chart
final_chart.add_annotation(text="Grey area: the middle half of the other 26 EU countries.<br>"
                                "Source: CSO (Ireland), Eurostat (other countries).",
                           xref="paper", yref="paper", x=0, y=-0.25, yanchor="top", align="left",
                           showarrow=False, font=dict(size=11, color="grey"))

# Hover, legend under the chart, space at top and bottom
final_chart.update_layout(hovermode="x unified",
                          hoverlabel_namelength=-1,
                          xaxis_hoverformat="%b %Y",
                          legend=dict(orientation="h", y=-0.15),
                          height=640,
                          margin=dict(t=120, b=160),
                          title_y=0.95)
final_chart.show()

What: Added crisis shading, a zero line, hover, and a note under the chart.

Why: The shading shows the crisis period, the zero line shows when prices fell, and the note explains the grey area and the data sources.

Result: A clean chart, and hovering on a month shows the Ireland and Eurozone average values.

### 2.8 Save the final chart

In [9]:
# Source: Plotly HTML export, https://plotly.com/python/interactive-html-export/

final_chart.write_html("price_chart.html", include_plotlyjs=True)

What: Saved the final chart as an HTML file.

Why: So it opens in any browser, even offline.

Result: A price_chart.html with all 4 buttons and hover working.